<a id=0></a>
# 7.Categorical Features
カテゴリカル特徴量（変数）の取り扱い

---
### [1.LabelEncoder()](#1)
### [2.get_dummies()](#2)
### [3.OneHotEncoder()](#3)
### [4.pd.get_dummies()とOneHotEncoder()の違い](#4)
### [5.Seriesのstr属性を使う](#5)

---

データセットとしてsample1_without_index.csvを使用する

In [65]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [66]:
df = pd.read_csv('sample1_without_index.csv')
df.head()

,Date,Price,Quantity,Width,Height,Quality,Score,Difference,Color,Shape
0,1997-07-05,2291,25,2.946650,5.305868,45.893300,52.762659,0.276266,green,triangle
1,1997-07-06,506,16,1.915208,0.679004,50.611735,31.453719,-1.854628,blue,NaN
2,1997-07-07,9629,32,7.869855,6.563335,43.830416,56.239011,0.623901,blue,square
3,1997-07-08,6161,67,6.375209,5.756029,41.358007,61.453113,1.145311,green,square
4,NaN,8570,55,0.390629,3.578136,55.739709,NaN,1.037190,red,square


In [67]:
df = df[['Color', 'Shape']] # 必要な列だけ、今回はColorとShapeを抽出

In [68]:
df.head()

,Color,Shape
0,green,triangle
1,blue,NaN
2,blue,square
3,green,square
4,red,square


In [69]:
df.isnull().sum()

Color    4
Shape    5
dtype: int64

In [70]:
df[df['Color'].isnull()].index # Color列に欠損値がある行のインデックスを取得

Index([19, 37, 40, 73], dtype='int64')

---
<a id=1></a>
[Topへ](#0)

---
## 1. LabelEncoder()  
https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelEncoder.html  
※ ラベルを数値(0, 1, 2, ...)で置換する

注意！　目的変数に使用すること。inputには使用しない。数値によって別の関係性があると誤解する。

In [71]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
encoder.fit(df['Color']) # Color列のユニークな値を学習させる
encoder.classes_ # 学習したユニークな値を確認する

array(['blue', 'green', 'red', nan], dtype=object)

In [72]:
encoder.transform(df['Color']) # Color列の値を数値に変換する
df.head()

,Color,Shape
0,green,triangle
1,blue,NaN
2,blue,square
3,green,square
4,red,square


In [73]:
df_ce = df.copy() # dfのコピーを作成
df_ce['Color_encoded'] = encoder.transform(df['Color']) # Color列を数値に変換して新しい列に保存
df_ce = df_ce[['Color', 'Color_encoded', 'Shape']] # 必要な列だけ抽出
df_ce.loc[36:42] # 36行目から42行目までを表示、欠損値がある行を確認する

,Color,Color_encoded,Shape
36,green,1,square
37,NaN,3,circle
38,blue,0,square
39,blue,0,square
40,NaN,3,triangle
41,red,2,NaN
42,green,1,square


---
<a id=2></a>
[Topへ](#0)

---
## 2. get_dummies()  
https://pandas.pydata.org/docs/reference/api/pandas.get_dummies.html  
※　カテゴリー変数をダミー変数化（0 or 1）する

* ダミー変数化を実行
* drop_first=Trueとは
* np.nanはどうなるのか
---

ダミー変数化を実行

In [74]:
pd.get_dummies(df['Color']).head() # Color列をダミー変数化する

,blue,green,red
0,False,True,False
1,True,False,False
2,True,False,False
3,False,True,False
4,False,False,True


drop_first=Trueとは  

In [75]:
pd.get_dummies(df['Color'], drop_first=True).head() # Color列をダミー変数化する、drop_first=Trueで最初の列を削除して多重共線性を回避する
# どちらもFalseがBlueとなるため、Blueを基準にしてRedとGreenの有無を表すことになる
# Drop fistを使うのが定石

,green,red
0,True,False
1,False,False
2,False,False
3,True,False
4,False,True


In [76]:
df_cd = pd.get_dummies(df, columns=['Color'], drop_first=True) # Color列をダミー変数化して、元のColor列は削除する
df_cd.head()

,Shape,Color_green,Color_red
0,triangle,True,False
1,NaN,False,False
2,square,False,False
3,square,True,False
4,square,False,True


In [77]:
df_cd = pd.get_dummies(df, columns=['Color', 'Shape'], drop_first=True) # Color列とShape列をダミー変数化して、元の列は削除する
df_cd.head()

,Color_green,Color_red,Shape_square,Shape_triangle
0,True,False,False,True
1,False,False,False,False
2,False,False,True,False
3,True,False,True,False
4,False,True,True,False


np.nanはどうなるのか
NanはBlueとして認識されてしまう

In [78]:
# Nanを認識できるようにする
df_cd = pd.get_dummies(df, columns=['Color', 'Shape'], drop_first=True, dummy_na=True) # Color列とShape列をダミー変数化して、元の列は削除する、dummy_na=Trueで欠損値もダミー変数化する
df_cd.head()

,Color_green,Color_red,Color_nan,Shape_square,Shape_triangle,Shape_nan
0,True,False,False,False,True,False
1,False,False,False,False,False,True
2,False,False,False,True,False,False
3,True,False,False,True,False,False
4,False,True,False,True,False,False


---
<a id=3></a>
[Topへ](#0)

---
## 3. OneHotEncoder()  
※　One-hot : ひとつが1で他は0  
※　pd.get_dummies()にはない機能を使ってダミー変数化を行う

https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html

デフォルトのKeyword Argument : drop=None, handle_unknown='error'

* OneHotEncoder()を使ってみる
* 複数の特徴量を変換
---

OneHotEncoder()を使ってみる

In [79]:
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder()

In [80]:
encoder.fit(df[['Color']]) # Color列のユニークな値を学習させる
encoder.categories_ # 学習したユニークな値を確認する

[array(['blue', 'green', 'red', nan], dtype=object)]

In [81]:
encoder.transform(df[['Color']]).toarray()[:5] # Color列の値を数値に変換する、toarray()で配列に変換する

array([[0., 1., 0., 0.],
       [1., 0., 0., 0.],
       [1., 0., 0., 0.],
       [0., 1., 0., 0.],
       [0., 0., 1., 0.]])

複数の特徴量を変換

In [82]:
encoder = OneHotEncoder()
encoder.fit(df) # Color列とShape列のユニークな値を学習させる
encoder.categories_


[array(['blue', 'green', 'red', nan], dtype=object),
 array(['circle', 'square', 'triangle', nan], dtype=object)]

In [83]:
encoder.transform((df)).toarray()[:5] # Color列とShape列の値を数値に変換する、toarray()で配列に変換する

array([[0., 1., 0., 0., 0., 0., 1., 0.],
       [1., 0., 0., 0., 0., 0., 0., 1.],
       [1., 0., 0., 0., 0., 1., 0., 0.],
       [0., 1., 0., 0., 0., 1., 0., 0.],
       [0., 0., 1., 0., 0., 1., 0., 0.]])

---
<a id=4></a>
[Topへ](#0)

---
## 4. pd.get_dummies()とOneHotEncoder()の違い

* get_dummies()ではトレインセットとテストセットに差が生じる
* OneHotEncoder(handle_unknown='error', drop='first')の場合
* OneHotEncoder(handle_unknown='ignore')の場合
---

get_dummies()ではトレインセットとテストセットに差が生じる

In [84]:
# Random seed1を設定
np.random.seed(1)
s = pd.Series(np.random.choice([0, 1], size=len(df)), name='target') # 0と1をランダムにdfの行数分生成する
s[:10] # 最初の10個を表示する

0    1
1    1
2    0
3    0
4    1
5    1
6    1
7    1
8    1
9    0
Name: target, dtype: int64

In [85]:
df_new = pd.concat([df, s], axis=1) # dfとsを横方向に結合する
df_new.head()

,Color,Shape,target
0,green,triangle,1
1,blue,NaN,1
2,blue,square,0
3,green,square,0
4,red,square,1


In [86]:
from sklearn.model_selection import train_test_split
y = df_new.pop('target') # 目的変数をyに保存する
X = df_new # 説明変数をxに保存する

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.05, stratify=y, random_state=17)

In [92]:
encoder = OneHotEncoder()
encoder.fit_transform(X_train).toarray()[:10] # 学習データのColor列とShape列のユニークな値を学習させる

array([[0., 0., 1., 0., 0., 1., 0., 0.],
       [0., 1., 0., 0., 0., 1., 0., 0.],
       [0., 1., 0., 0., 0., 0., 1., 0.],
       [0., 1., 0., 0., 0., 1., 0., 0.],
       [0., 0., 1., 0., 0., 0., 1., 0.],
       [0., 0., 1., 0., 1., 0., 0., 0.],
       [1., 0., 0., 0., 0., 1., 0., 0.],
       [0., 0., 1., 0., 0., 0., 1., 0.],
       [0., 0., 1., 0., 1., 0., 0., 0.],
       [0., 1., 0., 0., 1., 0., 0., 0.]])

OneHotEncoder(handle_unknown='error', drop='first')の場合

OneHotEncoder(handle_unknown='ignore')の場合

#### 状況に応じて使い分ける（例）

Train dataにはあって、Test dataにはない値が出てくる場合がある、逆に
サンプル数が少なく、カテゴリが多い場合には、Test dataにはあってTrain dataにはない値といったものもありうる


* 分類される値が少ない、レコード量が多い  
    ＝＞　testデータに欠ける値はない　＝＞　get_dummies, OneHotEncoder(drop='first')
* 分類される値が少ない、レコード量が少ない  
    ＝＞　testデータに欠ける値があるかもしれない　＝＞　OneHotEncoder(handle_unknown='error', drop='first')
* 分類される値が多い、レコード量が少ない  
    ＝＞　testデータにtrainデータにない値が確実に入る　＝＞ OneHotEncoder, handle_unknown='ignore'

---
<a id=5></a>
[Topへ](#0)

---
## 5.Seriesのstr属性を使う

* Series.strとは
* メソッドを確認
* 利用頻度の高い置換、抽出、分離
---

Series.strとは

In [87]:
df = pd.DataFrame()
df['ID'] = ['A-123', 'B-456', 'A-789', 'B-123']
df['Color'] = ['py/white black', 'red green blue', 'py/yellow', 'purple white']
df

,ID,Color
0,A-123,py/white black
1,B-456,red green blue
2,A-789,py/yellow
3,B-123,purple white


メソッドを確認

利用頻度の高い置換、抽出、分離

---
[Topへ](#0)

---
## 以上
    
---